# 🎙️ Unified Noise Suppression & Speaker Diarization Pipeline

This notebook provides a production-grade, end-to-end pipeline that takes raw audio files from Google Drive, performs **dynamic noise reduction**, saves the cleaned audio to a specified folder, and runs **speaker diarization** (identifying who spoke when) using **Pyannote 4.x**.

---

### 🌟 Workflow Sequence
1. **Audio Ingestion**: Loads audio files from Google Drive.
2. **Denoising & Resampling**: Standardizes the audio in memory to a **16kHz mono WAV** format and suppresses non-stationary background noise.
3. **File Export (Denoised)**: Saves the clean WAV audio to a dedicated cleaned audio directory.
4. **Pyannote Diarization**: Runs deep-learning-based speaker diarization on the cleaned audio file (utilizing T4 GPU acceleration).
5. **Timeline Refinement**: Merges consecutive turns from the same speaker separated by small silence gaps.
6. **Multi-Format Serialization**: Saves timelines to CSV, JSON, RTTM, a premium Markdown report, and a **plain text timestamp file**.

## ⚙️ Step 0: Setup & Core Dependencies

Make sure your Google Colab runtime is set to **T4 GPU** (*Runtime -> Change runtime type -> T4 GPU*).

Run the cell below to install the necessary packages. It uses version pins and settings designed to avoid conflict errors in Google Colab's Python 3.12 environment.

In [ ]:
# 1. Uninstall pre-installed incompatible torchao to prevent import conflicts with PyTorch
!pip uninstall -y -q torchao

# 2. Install Pyannote 4.x, noisereduce (version 3.0.3+ for NumPy 2.x support), and audio libraries
# This keeps Colab's default NumPy 2.x active and fully compatible with PyTorch/CUDA wheels
!pip install -q "pyannote.audio>=4.0.1" "noisereduce>=3.0.3" librosa soundfile pandas scipy --prefer-binary

print("\n[SUCCESS] Core dependencies installed successfully!")
print("[IMPORTANT] You MUST click 'RESTART SESSION' or 'RESTART RUNTIME' in the Colab menu (Runtime -> Restart session) for the newly installed packages to load cleanly!")

## 🔑 Step 1: Configuration & Hugging Face Access

Before running the cells below, configure your Hugging Face token:
1. Go to your Hugging Face settings and create a **Read Access Token**.
2. Visit **[pyannote/speaker-diarization-community-1](https://huggingface.co/pyannote/speaker-diarization-community-1)** and accept the user conditions.
3. In Google Colab, open the Secrets manager (the key icon 🔑 in the left sidebar).
4. Add a secret named **`HF_TOKEN`** and paste your token.
5. Toggle the **"Notebook access"** switch to **ON** for this secret!

In [ ]:
import torch
from google.colab import userdata

# Retrieve HF Token
try:
    hf_token = userdata.get('HF_TOKEN')
    print("[SUCCESS] Retrieved Hugging Face access token (HF_TOKEN) from Secrets.")
except Exception:
    hf_token = None
    print("[WARNING] 'HF_TOKEN' not found in Secrets. If Pyannote loading fails, please configure it.")

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {device}")
if device.type == "cpu":
    print("[WARNING] Running on CPU. This will be SIGNIFICANTLY slower. Please switch your runtime to T4 GPU (Runtime -> Change runtime type).")

## 🎵 Step 2: Interactive Parameter Configurations (Colab Forms)

Specify the path to your raw audio file and configure output directories. 

### 💡 Input Path Reference:
- **Google Drive File**: Mount Drive and enter a path starting with `/content/drive/MyDrive/` (e.g. `/content/drive/MyDrive/MyAudioFiles/interview.m4a`).
- **Local Upload**: Upload a file directly to the Colab sidebar using the folder icon and enter a path starting with `/content/` (e.g. `/content/recording.wav`).

The notebook automatically mounts Google Drive and sets up the folders for storing the results.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive (Required if loading files from your Drive folders)
print("Mounting Google Drive...")
try:
    drive.mount('/content/drive')
    print("[SUCCESS] Google Drive mounted.")
except Exception as e:
    print("[INFO] Drive mount skipped or already mounted.")

# @markdown ### 📁 File Path Parameters
# @markdown Specify the path to your source audio file and folder locations:
input_audio_path = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Sample Audio Files/MarauliKhurad1.m4a" # @param {type:"string"}
cleaned_audio_folder = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Noise Suppression And Diarization /Cleaned_Audio/" # @param {type:"string"}
diarization_output_folder = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Noise Suppression And Diarization /Diarization_Outputs/" # @param {type:"string"}


# @markdown ### 👥 Diarization Clustering Constraints (Optional)
# @markdown Set to 0 to let Pyannote estimate the number of speakers automatically (highly recommended for generic files).
num_speakers = 0 # @param {type:"integer"}
min_speakers = 0 # @param {type:"integer"}
max_speakers = 0 # @param {type:"integer"}

# @markdown ### 📅 Timeline Post-Processing
# @markdown Merge adjacent turns of the same speaker if the silence gap is less than this threshold (seconds):
max_merge_gap = 1.5 # @param {type:"number"}

# Verify input file existence
if not os.path.exists(input_audio_path):
    print(f"\n[ERROR] File not found at: '{input_audio_path}'")
    print("Please check the path inside your Google Drive or local file explorer and run this cell again.")
else:
    # Extract filename information for dynamic naming
    audio_filename = os.path.basename(input_audio_path)
    audio_name_only, _ = os.path.splitext(audio_filename)
    
    # Create output directories if they don't exist
    os.makedirs(cleaned_audio_folder, exist_ok=True)
    os.makedirs(diarization_output_folder, exist_ok=True)
    
    print(f"\n[SUCCESS] Input file verified: '{input_audio_path}'")
    print(f"[INFO] Cleaned audio will be saved to: '{os.path.join(cleaned_audio_folder, f"{audio_name_only}_cleaned.wav")}'")
    print(f"[INFO] Diarization reports and logs will be saved to: '{diarization_output_folder}'")

## 🧹 Step 3: Audio Ingestion, 16kHz Standardization & Noise Suppression

This step loads your audio and converts it to a standard **16kHz mono WAV** in memory (using `librosa.load`). Standardizing the file is necessary because Pyannote's speaker models are trained on 16kHz single-channel recordings. 

Next, it runs non-stationary noise reduction (`noisereduce`) to strip away background chatter, hum, and hiss, and saves the cleaned wav file to your configured folder.

In [ ]:
import numpy as np
# Runtime compatibility patch for older SciPy versions loaded with NumPy 2.x
if not hasattr(np._core._multiarray_umath, "_blas_supports_fpe"):
    setattr(np._core._multiarray_umath, "_blas_supports_fpe", lambda x: False)

import librosa
import soundfile as sf
import noisereduce as nr

if 'audio_name_only' in locals() and os.path.exists(input_audio_path):
    print(f"Ingesting and resampling audio: '{input_audio_path}'")
    try:
        # Load and standardise to 16kHz mono directly in memory
        y, sr = librosa.load(input_audio_path, sr=16000, mono=True)
        print(f"[SUCCESS] Loaded audio. Sample Rate: {sr} Hz, Samples Count: {len(y)}")
        
        print("Applying Non-Stationary Noise Reduction...")
        # Apply noise reduction (stationary=False adapts to changing background chatter)
        y_denoised = nr.reduce_noise(y=y, sr=sr, stationary=False, prop_decrease=0.85)
        
        # Save clean WAV file to the user-specified folder path
        cleaned_audio_path = os.path.join(cleaned_audio_folder, f"{audio_name_only}_cleaned.wav")
        sf.write(cleaned_audio_path, y_denoised, sr)
        
        print(f"\n[SUCCESS] Noise suppression complete!")
        print(f"Cleaned audio successfully stored at: '{cleaned_audio_path}'")
    except Exception as e:
        print(f"[ERROR] Audio preprocessing or noise reduction failed: {e}")
        raise e
else:
    print("[ERROR] Please execute Step 2 successfully before running this cell.")

## 👥 Step 4: Run Speaker Diarization (Pyannote 4.x)

Load Pyannote's `community-1` pipeline and run it directly on the noise-suppressed, standardized audio file saved in Step 3.

In [ ]:
from pyannote.audio import Pipeline
from pyannote.audio.pipelines.utils.hook import ProgressHook

if 'cleaned_audio_path' in locals() and os.path.exists(cleaned_audio_path):
    print("Initializing Pyannote Speaker Diarization Pipeline...")
    try:
        diarization_pipeline = Pipeline.from_pretrained(
            "pyannote/speaker-diarization-community-1",
            token=hf_token
        )
        diarization_pipeline.to(device)
        print("[SUCCESS] Diarization pipeline loaded successfully.")
    except Exception as e:
        print(f"[ERROR] Failed to load Pyannote pipeline. Verify your HF_TOKEN is valid and that you accepted the user conditions.")
        raise e

    print("\n--- Running Speaker Diarization on Cleaned Audio ---")
    
    # Setup pipeline params
    diarization_params = {}
    if num_speakers > 0:
        diarization_params["num_speakers"] = num_speakers
    else:
        if min_speakers > 0:
            diarization_params["min_speakers"] = min_speakers
        if max_speakers > 0:
            diarization_params["max_speakers"] = max_speakers

    with ProgressHook() as hook:
        diarization_output = diarization_pipeline(cleaned_audio_path, hook=hook, **diarization_params)

    # Extract speaker segments
    raw_speaker_segments = []
    for turn, speaker in diarization_output.speaker_diarization:
        raw_speaker_segments.append({
            "start": turn.start,
            "end": turn.end,
            "speaker": speaker
        })

    print(f"\nDiarization complete. Identified {len(set(s['speaker'] for s in raw_speaker_segments))} unique speaker(s).")
    print(f"Total raw speaker turns: {len(raw_speaker_segments)}")
else:
    print("[ERROR] Cleaned audio file path not found. Make sure Step 3 completed successfully.")

## 📊 Step 5: Timeline Merging & Speaker Analytics

This step refines raw speaker segments by grouping adjacent speaker turns (if the silent gap between them is smaller than `max_merge_gap`). It also calculates speaker analytics including speech time duration, voice share percentage, and overall turn counts.

In [ ]:
if 'raw_speaker_segments' in locals() and raw_speaker_segments:
    # Merges consecutive speaker segments of the same speaker if the gap between them is small
    def merge_speaker_segments(segments, max_gap=1.5):
        if not segments:
            return []
        sorted_segs = sorted(segments, key=lambda x: x["start"])
        merged = []
        current = sorted_segs[0].copy()
        for next_seg in sorted_segs[1:]:
            if next_seg["speaker"] == current["speaker"] and (next_seg["start"] - current["end"]) <= max_gap:
                current["end"] = max(current["end"], next_seg["end"])
            else:
                merged.append(current)
                current = next_seg.copy()
        merged.append(current)
        return merged

    # Process segments
    speaker_segments = merge_speaker_segments(raw_speaker_segments, max_gap=max_merge_gap)

    # Calculate speaker stats
    speaker_stats = {}
    total_speech_time = 0.0

    for seg in speaker_segments:
        duration = seg["end"] - seg["start"]
        speaker = seg["speaker"]
        total_speech_time += duration
        
        if speaker not in speaker_stats:
            speaker_stats[speaker] = {
                "total_duration": 0.0,
                "turn_count": 0
            }
        speaker_stats[speaker]["total_duration"] += duration
        speaker_stats[speaker]["turn_count"] += 1

    print(f"[SUCCESS] Speaker turns merged from {len(raw_speaker_segments)} down to {len(speaker_segments)} clean turns.")
    print("\n--- Speaker Statistics Summary ---")
    for spk, stats in speaker_stats.items():
        percentage = (stats["total_duration"] / total_speech_time) * 100 if total_speech_time > 0 else 0
        print(f"{spk}:")
        print(f"  - Total Speech Time: {stats['total_duration']:.2f}s ({percentage:.1f}%)")
        print(f"  - Turn Count: {stats['turn_count']}")
else:
    print("[ERROR] No speaker segments available. Please execute Step 4 first.")

## 💾 Step 6: Multi-Format Serialization & Reports

This step exports the speaker diarization timestamps and statistics into multiple formats inside your configured **`diarization_output_folder`**:
1. **Plain Text timestamps file** (`.txt`): A simple text log showing speaker turns and timestamps.
2. **Premium Markdown report** (`.md`): Structured statistics tables and a text-based speech participation chart.
3. **CSV Timeline** (`.csv`): Tabular layout for spreadsheets.
4. **JSON Timeline** (`.json`): Structured data file.
5. **RTTM Timeline** (`.rttm`): Rich Transcription Time Marker standard format.

In [ ]:
import csv
import json
import os
from IPython.display import Markdown, display

if 'speaker_segments' in locals() and speaker_segments:
    # Time formatting helper: convert seconds to [MM:SS.d] or [HH:MM:SS.d]
    def format_time(seconds):
        hours = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        secs = int(seconds % 60)
        millis = int((seconds - int(seconds)) * 10)
        if hours > 0:
            return f"{hours:02d}:{minutes:02d}:{secs:02d}.{millis:01d}"
        else:
            return f"{minutes:02d}:{secs:02d}.{millis:01d}"

    # NEW: Create a specific subfolder for this audio file's results
    specific_output_folder = os.path.join(diarization_output_folder, audio_name_only)
    os.makedirs(specific_output_folder, exist_ok=True)

    # Define output file paths based on the new specific_output_folder
    txt_output_path = os.path.join(specific_output_folder, f"{audio_name_only}_diarization_timestamps.txt")
    md_output_path = os.path.join(specific_output_folder, f"{audio_name_only}_diarization_report.md")
    csv_output_path = os.path.join(specific_output_folder, f"{audio_name_only}_timeline.csv")
    json_output_path = os.path.join(specific_output_folder, f"{audio_name_only}_timeline.json")
    rttm_path = os.path.join(specific_output_folder, f"{audio_name_only}_timeline.rttm")

    # 1. Export requested text file for speaker diarization timestamps
    with open(txt_output_path, "w", encoding="utf-8") as txt_file:
        txt_file.write(f"Speaker Diarization Timestamps for: {audio_filename}\n")
        txt_file.write("=" * 60 + "\n\n")
        for entry in speaker_segments:
            time_str = f"[{format_time(entry['start'])} - {format_time(entry['end'])}]"
            txt_file.write(f"{time_str} {entry['speaker']}\n")

    # 2. Export standard RTTM file (with URI sanitization)
    rttm_annotation_to_export = diarization_output.speaker_diarization.copy()
    rttm_annotation_to_export.uri = rttm_annotation_to_export.uri.replace(" ", "_")

    with open(rttm_path, "w") as rttm_file:
        rttm_annotation_to_export.write_rttm(rttm_file)

    # 3. Save JSON Timeline
    with open(json_output_path, "w", encoding="utf-8") as f:
        json.dump(speaker_segments, f, indent=4, ensure_ascii=False)

    # 4. Save CSV Timeline
    with open(csv_output_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Speaker", "Start_Time", "End_Time", "Timestamp_Formatted"])
        for entry in speaker_segments:
            writer.writerow([
                entry["speaker"],
                entry["start"],
                entry["end"],
                f"[{format_time(entry['start'])} - {format_time(entry['end'])}]"
            ])

    # 5. Generate Premium Markdown Report
    with open(md_output_path, "w", encoding="utf-8") as f:
        f.write(f"# 🎙️ Speaker Diarization Analysis Report: {audio_filename}\n\n")
        f.write("Generated using the Pyannote 4.x Speaker Diarization Pipeline (with dynamic noise reduction).\n\n")
        f.write("## 📊 Speaker Participation Summary\n\n")
        f.write("| Speaker | Total Speaking Duration | Turn Count | Speech % |\n")
        f.write("| :--- | :--- | :--- | :--- |\n")
        for spk, stats in sorted(speaker_stats.items()):
            percentage = (stats["total_duration"] / total_speech_time) * 100 if total_speech_time > 0 else 0
            f.write(f"| **{spk}**: | {stats['total_duration']:.2f}s | {stats['turn_count']} | {percentage:.1f}% |\n")

        f.write("\n### 📈 Visual Share of Speech\n\n")
        for spk, stats in sorted(speaker_stats.items()):
            percentage = (stats["total_duration"] / total_speech_time) * 100 if total_speech_time > 0 else 0
            bars = "█" * int(percentage // 4)
            f.write(f"- **{spk}**: `{bars}` ({percentage:.1f}%)\n")

        f.write("\n---\n\n## 📅 Cleaned Timeline\n\n")
        for entry in speaker_segments:
            time_str = f"[{format_time(entry['start'])} - {format_time(entry['end'])}]"
            f.write(f"> **{entry['speaker']}** `{time_str}`\n\n")
        f.write("---\n")

    print(f"\n[SUCCESS] Exported reports and timelines to '{specific_output_folder}':")
    print(f" - Plain Text Timestamps: {os.path.basename(txt_output_path)}")
    print(f" - Markdown Report: {os.path.basename(md_output_path)}")
    print(f" - Standard RTTM: {os.path.basename(rttm_path)}")
    print(f" - JSON Timeline: {os.path.basename(json_output_path)}")
    print(f" - CSV Timeline: {os.path.basename(csv_output_path)}")

    print("\n--- Diarization Report Preview ---")
    with open(md_output_path, "r") as f:
        display(Markdown(f.read()))
else:
    print("[ERROR] No speaker segments found to serialize.")